In [13]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

stats = pd.read_parquet(RAW / "player_stats_week_2020_2026.parquet")
schedules = pd.read_parquet(RAW / "schedules_2020_2026.parquet")
snaps = pd.read_parquet(RAW / "snap_counts_2020_2026.parquet")
ids = pd.read_parquet(RAW / "player_ids.parquet")

stats.shape

(119897, 150)

In [8]:
before = len(stats)

stats = stats[stats["position"].isin(["QB", "RB", "WR", "TE"])]   # fantasy positions only
stats = stats[stats["season_type"] == "REG"]                       # regular season only
stats = stats[stats["player_id"].notna()]                          # drop blank junk rows

print(f"{before:,} rows -> {len(stats):,} rows")
stats["position"].value_counts()

119,897 rows -> 37,313 rows


position
WR    15606
RB     9764
TE     7800
QB     4143
Name: count, dtype: int64

In [9]:
snaps = snaps[snaps["game_type"] == "REG"]          # regular season only, to match stats
ids = ids.drop_duplicates("pfr_id")                  # one row per player, so the merge doesn't duplicate rows

snaps = snaps.merge(ids, left_on="pfr_player_id", right_on="pfr_id", how="left")

print("snap rows with no GSIS match:", snaps["gsis_id"].isna().sum())

snap rows with no GSIS match: 185


In [10]:
snaps = snaps[["gsis_id", "season", "week", "offense_snaps", "offense_pct"]]
snaps = snaps.rename(columns={"gsis_id": "player_id"})            # same name as in stats
snaps = snaps.dropna(subset=["player_id"]).drop_duplicates(["player_id", "season", "week"])

stats = stats.merge(snaps, on=["player_id", "season", "week"], how="left", validate="many_to_one")

print("player-weeks with no snap data:", stats["offense_pct"].isna().sum())
stats[["player_display_name", "position", "season", "week", "offense_pct"]].head()

player-weeks with no snap data: 38


,player_display_name,position,season,week,offense_pct
0,Tom Brady,QB,2020,1,1.00
1,Drew Brees,QB,2020,1,0.97
2,Jason Witten,TE,2020,1,0.44
3,Larry Fitzgerald,WR,2020,1,0.79
4,Ben Roethlisberger,QB,2020,1,1.00


In [11]:
import numpy as np

games = schedules[["game_id", "home_team", "away_team",
                   "total_line", "spread_line", "home_rest", "away_rest"]]

stats = stats.merge(games, on="game_id", how="left", validate="many_to_one")

home = stats["team"] == stats["home_team"]                    # True if the player's team is at home

stats["is_home"] = home.astype(int)                            # 1 = home, 0 = away
stats["rest_days"] = np.where(home, stats["home_rest"], stats["away_rest"])
stats["implied_total"] = np.where(home,
                                  (stats["total_line"] + stats["spread_line"]) / 2,
                                  (stats["total_line"] - stats["spread_line"]) / 2)

print("rows with no Vegas line:", stats["total_line"].isna().sum())
print("share of home games:", round(stats["is_home"].mean(), 3))
stats[["player_display_name", "team", "home_team", "is_home", "implied_total", "rest_days"]].head()

rows with no Vegas line: 0
share of home games: 0.5


,player_display_name,team,home_team,is_home,implied_total,rest_days
0,Tom Brady,TB,NO,0,22.25,7
1,Drew Brees,NO,NO,1,26.25,7
2,Jason Witten,LV,CAR,0,25.50,7
3,Larry Fitzgerald,ARI,SF,0,20.75,7
4,Ben Roethlisberger,PIT,NYG,0,25.00,7


In [12]:
stats = stats.reset_index(drop=True)      # tidy row numbers after filtering

stats.to_parquet(PROCESSED / "player_weeks_clean.parquet", index=False)

print(f"Saved {len(stats):,} rows x {stats.shape[1]} columns")

Saved 37,313 rows x 161 columns
